# Extract words from YouTube videos

When learning Swedish independently, children's educational videos are actually every effective in my case. In the same time, it is important to review one or two times for the purpose of memorising. However, rewatching a long slow video is less convenient and often make poeple lose patience, my strategry is extracting these key words and later convert them to illustrational flashcards in my [En Språk Resa](https://www.ensprakresa.com/) platform.

This tutorial converts the speech into text and extracts the example nouns into a reusable table.

We aim to obtain the final result in two columns:

| letter | word |
|---|---|
| A | anka |
| ... | ... |

This notebook demonstrates two extraction methods:

1. **Regular expression extraction** — it summarise the moisy trascripts and find the best logics.
2. **OpenAI API extraction** — it use ai api to extract infomation that we need.

The workflow is:

- YouTube video
    ↓
- download audio
    ↓
- transcrib Swedish speech automatically by KB-Whisper
    ↓
- timestamped transcript DataFrame
    ↓
- regex extraction  OR  OpenAI API extraction
    ↓
- two-column letter/word DataFrame
    ↓
- CSV file

The example YouTube video is from: https://www.youtube.com/watch?v=pwiyMIFLDpg

## 1. 🍀 Create and activate a Python environment

Run these commands in Terminal from the project folder:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
```

Skip these commands when you are already working inside a suitable virtual environment.


## 2. 🍀 Install the required tools

Install the Python packages:

```bash
python -m pip install --upgrade \
  yt-dlp \
  transformers \
  torch \
  librosa \
  soundfile \
  pandas \
  openai \
  pydantic \
  python-dotenv
```

On macOS, install the command-line tools with Homebrew:

```bash
brew install ffmpeg deno
```

Check that they are available:

```bash
yt-dlp --version
ffmpeg -version
deno --version
```

### Why these tools are needed

- `yt-dlp` downloads the media.
- `ffmpeg` extracts and converts the audio.
- `deno` helps `yt-dlp` handle YouTube's JavaScript-based checks.
- `transformers` loads KB-Whisper.
- `torch` runs the speech-recognition model.
- `pandas` stores the transcript and extracted vocabulary.
- `openai` sends the transcript to an OpenAI model.
- `pydantic` defines the required structured response.
- `python-dotenv` loads the API key from a local `.env` file.


## 3. 🍀 Download only the audio

The following command worked on macOS with Chrome cookies:

```bash
yt-dlp \
  --cookies-from-browser chrome \
  --js-runtimes deno \
  --remote-components ejs:github \
  -x \
  --audio-format mp3 \
  -o "alphabet_audio.%(ext)s" \
  "https://www.youtube.com/watch?v=pwiyMIFLDpg"
```

### Explanation of the options

| Option | Meaning |
|---|---|
| `--cookies-from-browser chrome` | Uses the current Chrome session and cookies |
| `--js-runtimes deno` | Uses Deno as the JavaScript runtime |
| `--remote-components ejs:github` | Loads YouTube challenge-handling components |
| `-x` | Extracts audio only |
| `--audio-format mp3` | Converts the audio to MP3 |
| `-o "alphabet_audio.%(ext)s"` | Saves the result as `alphabet_audio.mp3` |

macOS may ask for permission to access **Chrome Safe Storage** in Keychain.
Enter the Mac login password and choose **Allow**.


## 4. 🍀 Confirm that the audio file exists


In [213]:
from pathlib import Path

audio_path = Path("alphabet_audio.mp3")

if not audio_path.is_file():
    raise FileNotFoundError(
        f"Audio file not found: {audio_path.resolve()}"
    )

print("Audio file:", audio_path.resolve())
print(
    "Size:",
    round(audio_path.stat().st_size / 1_000_000, 2),
    "MB",
)


Audio file: /Users/changliu/Documents/data-lab/tool/alphabet_audio.mp3
Size: 8.34 MB


## 5. 🍀 Transcribe Swedish speech with KB-Whisper Medium

For this video, `KBLab/kb-whisper-medium` produced more reliable Swedish vocabulary than the other tested models. A more genral powerfull model, such as 'WhisperModel', can recognise 'som i' but performed poorly in recognising swedish letter or words, e.g. 'e','ä', 'maskros.


In [214]:
from time import perf_counter
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition",
    model="KBLab/kb-whisper-medium",
)

start = perf_counter()

result = transcriber(
    str(audio_path),
    generate_kwargs={
        "language": "sv",
        "task": "transcribe",
    },
    return_timestamps=True,
)

elapsed = perf_counter() - start

print(f"Runtime: {elapsed:.1f} seconds")
print(result["text"])


Loading weights:   0%|          | 0/948 [00:00<?, ?it/s]

Runtime: 192.1 seconds
 Hej allihopa! I dag ska vi titta på alla bokstäver i alfabetet och vilka ord man kan använda bokstäverna i. Häng med! Alfabetet börjar med bokstaven A. Så här ser stora A ut och så här ser lilla A ut. Här kommer några ord som börjar. på a. A som i anka. A som i apelsin. A som i apa. A som i ankare. A som i ananas. A som i amborre. Bokstaven b. Stora B och lilla b. B som i, mm. BÅT. B som i BASG focusing. Banan B som i Bil B som i Bok B som i broccoli. Efter B så kommer bokstaven C och lilla c. C som i cykel C som i... citron C som i... clementin C som i... champinjon. C som i ... choklad. Bokstaven D. Stora D och lilla D. D som i .... daggmask. D som i ... Delfin Det är som i dörr Det är som i dragkedja. Det är som i dator! Stora e och lilla e. E som i ekorre. E som i eld. E som i elefant. E som i enhörning. Efter e kommer bokstaven f, stora f och lilla f. F som i får. Nää! F som i flaska. fågel F som i fjäril F som i fjäder Sen har vi bokstaven G. Stora G och l

### Inspect the returned object

`result` is a dictionary. Its main fields are:

- `result["text"]` — the complete transcription;
- `result["chunks"]` — shorter pieces of text with timestamps.


In [215]:
print(type(result))
print(result.keys())
print(result["chunks"][:2])

<class 'dict'>
dict_keys(['text', 'chunks'])
[{'timestamp': (0.0, 8.56), 'text': ' Hej allihopa!'}, {'timestamp': (9.48, 16.96), 'text': ' I dag ska vi titta på alla bokstäver i alfabetet och vilka ord man kan använda bokstäverna i.'}]


## 6. 🍀 Store the timestamped transcript in a DataFrame


In [216]:
import pandas as pd

transcript_rows = []

for chunk in result["chunks"]:
    start_time, end_time = chunk["timestamp"]

    transcript_rows.append(
        {
            "start": start_time,
            "end": end_time,
            "text": chunk["text"].strip(),
        }
    )

transcript_df = pd.DataFrame(transcript_rows)
print(transcript_df.shape)
transcript_df.loc[105:110, :]


(220, 3)


,start,end,text
105,560.04,561.66,och lilla M.
106,563.00,568.00,M som i måne.
107,569.00,573.02,M som i mus.
108,576.80,580.80,M som i maskros.
109,582.88,587.90,M som i morot.
110,593.00,599.00,Makaroner


## 7. 🍀 Method 1 — extract words with a regular expression

This method searches for the first word after the phrase `som i`, such as `A some i anka`.

It deliberately ignores the spoken letter before `som i`, because a speech recognition model may hear the spoken letter **N** as the Swedish word **en**.

Examples that should still work:

```text
A som i anka
En som i nalle
C som i... champinjon
P som i, päron
Ö som i ödla
```


In [219]:
import re

transcript_text = result["text"]

pattern = re.compile(
    r"\bsom\s+i\b[\s,.:;…-]*"
    r"([A-Za-zÅÄÖåäö]+(?:-[A-Za-zÅÄÖåäö]+)*)",
    flags=re.IGNORECASE,
)

raw_words = pattern.findall(transcript_text)

candidate_df = pd.DataFrame({"raw_word": raw_words})

candidate_df["letter"] = candidate_df["raw_word"].str[0].str.upper()

candidate_df["word"] = candidate_df["raw_word"].str.strip().str.lower()

print("Number of words found:", candidate_df["word"].nunique())
print(candidate_df[candidate_df["word"].duplicated(keep=False)])

candidate_df = candidate_df[~candidate_df["word"].duplicated(keep=False)]

candidate_df.loc[105:110, :]


Number of words found: 122
    raw_word letter      word
79  regnbåge      R  regnbåge
80  regnbåge      R  regnbåge


,raw_word,letter,word
105,jo,J,jo
106,yxa,Y,yxa
107,yoga,Y,yoga
108,Zucchini,Z,zucchini
109,zebra,Z,zebra
110,ål,Å,ål


### 7.1 How the regular expression works

Python automatically joins the two adjacent raw strings. The complete pattern is:

```regex
\bsom\s+i\b[\s,.:;…-]*([A-Za-zÅÄÖåäö]+(?:-[A-Za-zÅÄÖåäö]+)*)
```

- `\bsom\s+i\b`

    The two `\b` markers and the expression between them work together to match
    `som i` as complete words:

    ```text
    \b som ... i \b
    │             │
    start         end
    boundary      boundary
    ```

    - the first `\b` requires a word boundary before `som`;
    - `\s+` requires one or more whitespace characters between `som` and `i`;
    - the second `\b` requires a word boundary after `i`.

    Therefore, the pattern can match `som i`, but it does not treat the `som` inside `midsommar` or the `i` at the beginning of `igelkott` as separate words.

- `[\s,.:;…-]*`

    The square brackets define allowed separator characters after `som i`:

    - `\s` — whitespace;
    - `,` — comma;
    - `.` — full stop;
    - `:` — colon;
    - `;` — semicolon;
    - `…` — ellipsis;
    - `-` — hyphen.

    The final `*` means **zero or more** separator characters.

- `[A-Za-zÅÄÖåäö]+`

    The character set matches one letter at a time:

    - `A-Z`
    - `a-z`
    - `ÅÄÖ`
    - `åäö`

    The `+` means **one or more consecutive letters**, so the complete expression can match a word such as `anka`, `äpple`, or `ödla`.

- `(?:-[A-Za-zÅÄÖåäö]+)*`

    This optional non-capturing group allows one or more hyphenated word sections. It makes the complete capturing group able to match:

    ```text
    anka
    U-båt
    walkie-talkie
    WC-toalett
    ```

- `flags=re.IGNORECASE`

    This makes `som i`, `Som i`, and `SOM I` equivalent for matching.


### 7.2 Create the two-column regex DataFrame

The letter is derived from the first character of the extracted word. 


In [220]:
regex_swedish_df = candidate_df.drop(columns=["raw_word"]).reset_index(drop=True)

print("Number of letters found:", regex_swedish_df["letter"].unique())
print("Number of words found:", regex_swedish_df["word"].unique())


Number of letters found: <StringArray>
['A', 'M', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'N', 'O',
 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'Å', 'Ä', 'Ö']
Length: 29, dtype: str
Number of words found: <StringArray>
[   'anka', 'apelsin',     'apa',  'ankare',  'ananas', 'amborre',      'mm',
    'basg',     'bil',     'bok',
 ...
    'åska',   'äpple',     'älg',   'ärtor',     'ägg',   'ängel',     'öga',
     'öra',    'ödla',    'öken']
Length: 121, dtype: str


### Limitation of the regex method

The speech-rocognition model is not 100% accurate. Therefore, the transcript does not always follow the patter `A som i anka`.

For example, in the transcript:

```text
F som i flaska. fågel F som i fjäril
```

The regex finds `flaska` and `fjäril`, but it misses `fågel`. It still need some mannual work.


## 8. 🍀 Method 2 — extract and normalize words with the OpenAI API

This method sends the complete transcript to a language model. The model can use the
alphabet order, surrounding examples, Swedish vocabulary, and the broader
sentence structure.

The model is asked to:

- identify all example nouns;
- normalize probable speech-recognition errors into standard Swedish spelling;
- preserve the original transcript wording for review;
- mark uncertain interpretations with lower confidence;


### 8.1 Store the API key safely

Create a `.env` file in the notebook's project folder:

```text
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-5.2
```

Add `.env` to `.gitignore`, never publish the API key in a notebook or Git repository.


In [221]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY was not found in the environment or .env file."
    )

print("OPENAI_API_KEY is available.")


OPENAI_API_KEY is available.


### 8.2 Define the required structured output


In [222]:
from typing import Literal
from pydantic import BaseModel


class AlphabetExample(BaseModel):
    letter: str
    word: str
    confidence: Literal["high", "medium", "low"]


class AlphabetExtraction(BaseModel):
    examples: list[AlphabetExample]


### 8.3 Send the complete transcript to the API

The prompt defines the task, but it does not provide a prepared list of correct
nouns. The model must inspect the transcript itself.


In [223]:
from openai import OpenAI

client = OpenAI()

response = client.responses.parse(
    model="gpt-5.2",
    reasoning={"effort": "medium"},
    input=[
        {
            "role": "system",
            "content": """
                        Extract and normalize the Swedish example words from the
                        speech-recognition transcript according to the swedish alphabet.

                        Desired output:
                            - letter		
                            - word	
                            - confidence

                        Each letter is followeed by several examples words starting with that letter. 
                        The transcript may contain misheard or misrecognized words.

                        Do not invent examples if you can not infer them from the transcript.
                        """,
        },
        {
            "role": "user",
            "content": transcript_text,
        },
    ],
    text_format=AlphabetExtraction,
)

api_result = response.output_parsed


### 8.4 Convert the detailed result to a review DataFrame


In [231]:
api_review_df = pd.DataFrame(
    example.model_dump()
    for example in api_result.examples
)

print("Rows returned:", len(api_review_df))
api_review_df.iloc[105:110, :]


Rows returned: 126


,letter,word,confidence
105,W,wc-toalett,high
106,W,wok,medium
107,X,xylofon,high
108,Y,yoyo,medium
109,Y,yxa,high


### 8.5 Review uncertain words


In [232]:
uncertain_rows = api_review_df.loc[
    api_review_df["confidence"] != "high"
]

print("Uncertain rows:", len(uncertain_rows))

uncertain_rows.head(20)


Uncertain rows: 6


,letter,word,confidence
5,A,abborre,medium
25,F,får,medium
43,I,igloo,medium
100,V,vas,medium
106,W,wok,medium
108,Y,yoyo,medium


In [233]:
api_swedish_df = api_review_df[["letter", "word"]]

print("Letters found:", api_swedish_df["letter"].unique())
print("Swedish words:", ", ".join(api_swedish_df["word"]))


Letters found: <StringArray>
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O',
 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'Å', 'Ä', 'Ö']
Length: 29, dtype: str
Swedish words: anka, apelsin, apa, ankare, ananas, abborre, båt, banan, bil, bok, broccoli, cykel, citron, clementin, champinjon, choklad, daggmask, delfin, dörr, dragkedja, dator, ekorre, eld, elefant, enhörning, får, flaska, fågel, fjäril, fjäder, gran, get, gaffel, gem, gurka, hare, haj, handduk, hammare, handväska, insekt, istapp, igelkott, igloo, jacka, juice, julgran, jordgubbe, jultomten, kiwi, katt, kaffe, kalv, kaktus, lampa, lamm, lim, lakrits, lego, måne, mus, maskros, morot, makaroner, nyckel, napp, nalle, nektarin, nypon, overall, oliv, oxe, orm, päron, planet, palm, pyjamas, pirat, quinoa, räv, rabarber, ring, robot, regnbåge, sol, sallad, snöflinga, sax, säl, tax, tejp, tand, tofflor, tandkräm, ubåt, ur, uggla, utomjording, vulkan, vatten, vas, vanilj, val, wienerbröd, walkie

## 9. 🍀 Alternative 3: Send the transcript to AI apps directly

The API costs a bit money, and the regex method takes a bit time. I have personally tested these three alternatives, and conclude that it is best to just send transcript to ChatGPT (since i pay monthly), and it can return back the correct alphabet with correct example nouns. I copied the results from `GPT-5.6 Sol Medium` as followed:

In [227]:
import pandas as pd

alphabet_words = {
    "A": "anka apelsin apa ankare ananas abborre",
    "B": "båt banan bil bok broccoli",
    "C": "cykel citron clementin champinjon choklad",
    "D": "daggmask delfin dörr dragkedja dator",
    "E": "ekorre eld elefant enhörning",
    "F": "får flaska fågel fjäril fjäder",
    "G": "gran get gaffel gem gurka",
    "H": "hare haj handduk hammare handväska",
    "I": "insekt istapp igelkott iglo",
    "J": "jacka juice julgran jordgubbe jultomten",
    "K": "kiwi katt kaffe kalv kaktus",
    "L": "lampa lamm lim lakrits lego",
    "M": "måne mus maskros morot makaroner",
    "N": "nyckel napp nalle nektarin nypon",
    "O": "overall oliv oxe orm",
    "P": "päron planet palm pyjamas pirat",
    "Q": "quinoa",
    "R": "räv rabarber ring robot regnbåge",
    "S": "sol sallad snöflinga sax säl",
    "T": "tax tejp tand tofflor tandkräm",
    "U": "u-båt ur uggla utomjording",
    "V": "vulkan vatten vas vanilj val",
    "W": "wienerbröd walkie-talkie wc-toalett wok",
    "X": "xylofon",
    "Y": "jojo yxa yoga",
    "Z": "zucchini zebra",
    "Å": "ål åtta åsna åska",
    "Ä": "äpple älg ärtor ägg ängel",
    "Ö": "öga öra ödla öken",
}

gpt56_swedish_df = (
    pd.Series(alphabet_words, name="word")
    .str.split()
    .explode()
    .rename_axis("letter")
    .reset_index()
)

## 10. 🍀 Comparision

The comparison identifies:

- words found only by the API, including examples missed because `som i` was
  omitted;
- words found only by regex, which may indicate an API omission or a literal
  transcription error requiring review.
- words processed by ChatGPT-5.6 sol directly

### 10.1 Import the manually corrected data


In [234]:
reference_swedish_df = pd.read_csv('swedish_alphabet_words_checked.csv', sep=';')
reference_swedish_df.loc[105:110, :]


,letter,word
105,W,wc-toalett
106,W,wok
107,X,xylofon
108,Y,yo-yo
109,Y,yxa
110,Y,yoga


### 10.2 Combine dataframes

Here we desing a heatmap to show the differences of each method compared with the reference

In [239]:
import pandas as pd

dataframes = {
    "reference": reference_swedish_df,
    "regex": regex_swedish_df,
    "api": api_swedish_df,
    "gpt56": gpt56_swedish_df,
}

combined_df = pd.concat(
    [
        df[["word"]].assign(source=source)
        for source, df in dataframes.items()
    ],
    ignore_index=True,
)

# Normalize words
combined_df["word"] = (
    combined_df["word"]
    .astype("string")
    .str.casefold()
)

# Convert to wide format
combined_df_wide = (
    combined_df
    .pivot_table(
        index="word",
        columns="source",
        values="word",
        aggfunc="first",
    )
    .reindex(columns=["reference", "regex", "api", "gpt56"])
)

combined_df_wide.columns.name = None

combined_df_wide

,reference,regex,api,gpt56
word,,,,
abborre,abborre,<NA>,abborre,abborre
amborre,<NA>,amborre,<NA>,<NA>
ananas,ananas,ananas,ananas,ananas
anka,anka,anka,anka,anka
ankare,ankare,ankare,ankare,ankare
...,...,...,...,...
åtta,åtta,<NA>,åtta,åtta
ödla,ödla,ödla,ödla,ödla
öga,öga,öga,öga,öga


### 10.3 Prepare data for visualisation

Divide data to different color groups:

- no data = 0 = grey
- wrong recognition = 1 = red
- correct recognition = 2 = green

`np.where` works like: 

```python
np.where(condition, value_if_true, value_if_false)
```

If the data is missing, set it to `0`, 
If the data is not missing: 1, if it is correct, set it to `2`, if it not correct, set it to `1`.

In [250]:
import numpy as np

# Put columns in the desired order
plot_df = combined_df_wide.copy()

# Whether the cell has a value (not NaN/NA) or not
present = plot_df.notna()

# True when this row represents a correct reference word
correct_word = plot_df["reference"].notna()
correct_word = correct_word.to_numpy()[:, None]

# Status:
# 0 = missing, 1 = wrong, 2 = correct
status = np.where(
    ~present,
    0,
    np.where(
        correct_word,
        2,
        1,
    ),
)
status[:5]

array([[2, 0, 2, 2],
       [0, 1, 0, 0],
       [2, 2, 2, 2],
       [2, 2, 2, 2],
       [2, 2, 2, 2]])

### 10.4 Heatmap visualisation

In [255]:
from IPython.display import display
import plotly.graph_objects as go
import plotly.io as pio

# Load Plotly.js from the internet instead of embedding it
pio.renderers.default = "notebook_connected"

# Text displayed inside each cell
cell_text = plot_df.fillna("").astype(str).to_numpy()

# Discrete colour scale:
# grey = missing, light red = wrong, light green = correct
colorscale = [
    [0.00, "#eeeeee"],
    [0.33, "#eeeeee"],
    [0.33, "#f8caca"],
    [0.66, "#f8caca"],
    [0.66, "#cce8cc"],
    [1.00, "#cce8cc"],
]

fig = go.Figure(
    go.Heatmap(
        z=status,
        x=["Reference", "Regex", "API", "GPT-5.6"],
        y=plot_df.index.astype(str),
        text=cell_text,
        texttemplate="%{text}",
        textfont={"size": 11},
        colorscale=colorscale,
        zmin=0,
        zmax=2,
        showscale=False,
        xgap=2,
        ygap=2,
        hovertemplate=(
            "Word: %{y}<br>"
            "Source: %{x}<br>"
            "Result: %{text}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    height=max(700, len(plot_df) * 24),
    width=850,
    xaxis_title="Recognition method",
    yaxis_title="Word variant",
    xaxis_side="top",
    yaxis_autorange="reversed",
    margin=dict(l=80, r=20, t=50, b=30),
)

display(fig)

### 10.5 Accuracy

In [256]:
methods = ["regex", "api", "gpt56"]

reference_present = plot_df["reference"].notna()
n_reference = reference_present.sum()

accuracy_table = pd.DataFrame(
    {
        method: {
            "Correct": (
                reference_present & plot_df[method].notna()
            ).sum(),
            "Wrong": (
                ~reference_present & plot_df[method].notna()
            ).sum(),
            "Missing": (
                reference_present & plot_df[method].isna()
            ).sum(),
        }
        for method in methods
    }
).T

accuracy_table["Accuracy (%)"] = (
    accuracy_table["Correct"] / n_reference * 100
).round(2)

accuracy_table

,Correct,Wrong,Missing,Accuracy (%)
regex,110,11,16,87.30
api,125,1,1,99.21
gpt56,123,3,3,97.62


Since the api used GPT5.2 medium, we interprete the results as: GPT5.2 is good enough for the informaiton extraction and correction.

## 11. 🍀 Conclusion

To extract required data from Youtube videos, we can follow the procedures:

- YouTube video
    ↓
- `yt-dlp` to download audio
    ↓
-  `KBLab/kb-whisper-medium` to transcribe speech
    ↓
- `AI` to correct transcript



### Note

Even when we use exactly the same model, procedure, and parameters, repeated runs may still produce slightly different results.